### Day 11: ReAct Agents — Reason + Act Loops

Today you learn the ReAct pattern — the foundation of all modern AI agents. Instead of one LLM call, agents loop: Think → Act → Observe → Think → Act... until the task is done.

#### 1. What Makes an Agent Different?

Chain (what you built so far):
  Input → Step1 → Step2 → Step3 → Output
  Fixed path, predetermined steps

Agent (what you build today):
  Input → Think → "I need tool X" → Use X → Observe result
         → Think → "I need tool Y" → Use Y → Observe result
         → Think → "I have enough info" → Final Answer
  Dynamic path, LLM decides next step every iteration

#### 2. The ReAct Loop

┌─────────────────────────────────────────────┐
│              ReAct Loop                     │
│                                             │
│  Question                                   │
│     ↓                                       │
│  REASON: "To answer this I need to..."      │
│     ↓                                       │
│  ACT: call_tool(name, args)                 │
│     ↓                                       │
│  OBSERVE: tool returned "..."               │
│     ↓                                       │
│  REASON: "Now I know X, but I still need Y" │
│     ↓                                       │
│  ACT: call_tool(name, args)                 │
│     ↓                                       │
│  OBSERVE: tool returned "..."               │
│     ↓                                       │
│  REASON: "I have all info needed"           │
│     ↓                                       │
│  Final Answer ✅                            │
└─────────────────────────────────────────────┘

In [ ]:
# Cell 1: Imports
from dotenv import load_dotenv
from langchain_groq import ChatGroq
# from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent
from typing import TypedDict, Annotated, List
import operator, json, math, os, requests
from datetime import datetime
from duckduckgo_search import DDGS

load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
print("✅ Ready")

✅ Ready


In [9]:
llm.invoke("what is the meaning of dihh in slang word?")

BadRequestError: Error code: 400 - {'error': {'message': 'The model `deepseek-r1-distill-llama-70b` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

In [8]:
# Cell 2: All tools our agent will use

@tool
def calculator(expression: str) -> str:
    """
    Evaluate mathematical expressions.
    Use for any arithmetic, percentages, or numerical calculations.
    Input: valid Python math expression like '100 * 0.18' or 'sqrt(144)'.
    """
    try:
        allowed = {
            'abs': abs, 'round': round, 'min': min,
            'max': max, 'pow': pow, 'sqrt': math.sqrt,
            'pi': math.pi, 'e': math.e, 'log': math.log
        }
        result = eval(expression, {"__builtins__": {}}, allowed)
        return str(result)
    except Exception as ex:
        return f"Calculation error: {ex}"


@tool
def web_search(query: str) -> str:
    """
    Search the web for current information.
    Use for recent facts, news, statistics, or anything
    that requires up-to-date knowledge.
    Input: a focused search query string.
    """
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=4))
        if not results:
            return "No results found."
        return "\n\n".join(
            f"Title: {r['title']}\nInfo: {r['body'][:300]}"
            for r in results
        )
    except Exception as e:
        return f"Search error: {e}"


@tool
def get_weather(city: str) -> str:
    """
    Get current weather for any city.
    Use when user asks about temperature, weather conditions,
    or climate in a specific location.
    Input: city name as a string.
    """
    try:
        geo = requests.get(
            f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1",
            timeout=5
        ).json()
        if not geo.get("results"):
            return f"City '{city}' not found"

        r = geo["results"][0]
        lat, lon = r["latitude"], r["longitude"]
        name = r["name"]
        country = r.get("country", "")

        weather = requests.get(
            f"https://api.open-meteo.com/v1/forecast?"
            f"latitude={lat}&longitude={lon}"
            f"&current=temperature_2m,weathercode,windspeed_10m,humidity",
            timeout=5
        ).json()

        c = weather["current"]
        return (
            f"{name}, {country}: "
            f"{c['temperature_2m']}°C, "
            f"Wind: {c['windspeed_10m']} km/h"
        )
    except Exception as e:
        return f"Weather error: {e}"


@tool
def get_current_time(timezone: str = "Asia/Kolkata") -> str:
    """
    Get current date and time.
    Use when user asks about current time, date, day, or year.
    Input: timezone string like 'Asia/Kolkata' or 'UTC'.
    """
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@tool
def wikipedia_search(topic: str) -> str:
    """
    Search Wikipedia for factual information about a topic.
    Use for historical facts, biographies, science concepts,
    or any topic needing encyclopedic knowledge.
    Input: topic or person to search for.
    """
    try:
        # Wikipedia API - free, no key needed
        url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + \
              topic.replace(" ", "_")
        resp = requests.get(url, timeout=5).json()

        if resp.get("type") == "disambiguation":
            return f"'{topic}' is ambiguous. Try a more specific term."

        title = resp.get("title", topic)
        extract = resp.get("extract", "No information found")
        return f"{title}: {extract[:500]}"
    except Exception as e:
        return f"Wikipedia error: {e}"


# All tools in one list
tools = [calculator, web_search, get_weather, get_current_time, wikipedia_search]
print(f"✅ {len(tools)} tools ready:")
for t in tools:
    print(f"   - {t.name}")

✅ 5 tools ready:
   - calculator
   - web_search
   - get_weather
   - get_current_time
   - wikipedia_search


In [3]:
# Cell 3: Easiest way - LangGraph's prebuilt ReAct agent
from langgraph.prebuilt import create_react_agent
from click import prompt

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="""You are a helpful AI assistant with access to tools.

    For every task:
    1. Think about what information you need
    2. Use the most appropriate tool
    3. Analyze the result
    4. Use more tools if needed
    5. Give a clear, complete final answer

    Always show your reasoning. Be thorough but concise."""
)

print("✅ ReAct agent created")

✅ ReAct agent created


C:\Users\Dhrumil Prajapati\AppData\Local\Temp\ipykernel_5880\2295929297.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [4]:
# Cell 4: Run the agent and see the full ReAct loop
def run_agent(question: str, verbose: bool = True):
    """Run agent and display full reasoning trace"""
    print(f"\n{'='*60}")
    print(f"❓ Question: {question}")
    print('='*60)

    result = agent.invoke({
        "messages": [HumanMessage(content=question)]
    })

    if verbose:
        print("\n📋 Full ReAct Trace:")
        for i, msg in enumerate(result["messages"]):
            if isinstance(msg, HumanMessage):
                print(f"\n[{i}] 👤 Human: {msg.content[:100]}")

            elif isinstance(msg, AIMessage):
                if msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"\n[{i}] 🧠 Reasoning → calling: {tc['name']}")
                        print(f"     Args: {tc['args']}")
                else:
                    print(f"\n[{i}] 🤖 Final Answer:")
                    print(f"     {msg.content}")

            elif isinstance(msg, ToolMessage):
                print(f"\n[{i}] 🔧 Tool Result ({msg.name}):")
                print(f"     {str(msg.content)[:200]}")

    # Return just the final answer
    final = result["messages"][-1].content
    return final


# Test simple queries
run_agent("What is 15 percantage of 85000?")


❓ Question: What is 15 percantage of 85000?


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': 'To find 15 percent of 85000, I will use the calculator function. \n\n<function=calculator {"expression": "85000 * 0.15"} </function>'}}

In [5]:
# Cell 5: Multi-step reasoning
run_agent(
    "Search for the population of Mumbai and calculate "
    "how many people that is per square kilometer "
    "given Mumbai is 603 square kilometers"
)


❓ Question: Search for the population of Mumbai and calculate how many people that is per square kilometer given Mumbai is 603 square kilometers


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=wikipedia_search{"topic": "population of Mumbai"}</function>\n\n'}}

In [6]:
# Cell 6: Multi-tool chaining
run_agent(
    "What is the current weather in Ahmedabad? "
    "Also tell me what time it is and if I should carry an umbrella."
)


❓ Question: What is the current weather in Ahmedabad? Also tell me what time it is and if I should carry an umbrella.


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=get_weather{"city": "Ahmedabad"}</function>\n'}}